In [ ]:
from importlib import reload
from adaptive_trade_extensions import make_ret_grid
import adaptive_reward_checkpoint_fresh
reload(adaptive_reward_checkpoint_fresh)

from adaptive_reward_checkpoint_fresh import run_adaptive_reward_yfinance_scheduled_flat_start_loop

live_res = run_adaptive_reward_yfinance_scheduled_flat_start_loop(
    checkpoint_path="output_adaptive_reward_live_TQQQ_20260611/live_checkpoint.joblib",
    output_dir="output_adaptive_reward_live_TQQQ_20260612",
    save_snapshot_path="output_adaptive_reward_live_TQQQ_20260612/live_checkpoint.joblib",

    start_date="2026-06-12",
    market_open_time="09:30",
    market_close_time="16:00",
    timezone="America/New_York",
    threshold_ret_grid_override=make_ret_grid(-0.5, 2.5, 0.05),

    local_data_dir="DataAPI/data",
    initial_capital=100000.0,
    fee_pct=0.0,
    poll_seconds=60,

    preopen_update=True,
    wait_until_start=True,
    verbose=True,
)

print("latest checkpoint:", live_res["latest_checkpoint_path"])
display(live_res["trades_df"].tail())
display(live_res["daily_log_df"].tail())

In [6]:
import pandas as pd
import numpy as np
from itertools import product
from pathlib import Path

SOURCE_DIR = Path("output_adaptive_reward_resumed_fresh_TQQQ_252days_at_2020")
OUTPUT_DIR = Path("output_replay_new_daily_gate_old_5m_signals_lookback30")

DAILY_REWARD_PATH = SOURCE_DIR / "daily_reward_log.csv"
OLD_TRADES_PATH = SOURCE_DIR / "trades.csv"
FIVE_MIN_PATH = Path("DataAPI/data/TQQQ_5M.csv")

INITIAL_CAPITAL = 100000.0
FEE_PCT = 0.0

LOOKBACK_DAYS = 252
MIN_OBS = 60
MIN_GAP = 0.02

def action_from_p(p_val, buy_level, sell_level):
    if not np.isfinite(p_val):
        return "NO_P"
    if p_val <= buy_level:
        return "FORCE_BUY"
    if p_val >= sell_level:
        return "FORCE_SELL"
    return "FREE"

def build_daily_gate_replay():
    df = pd.read_csv(DAILY_REWARD_PATH)
    df["date"] = pd.to_datetime(df["date"]).dt.normalize()
    df["p_day"] = pd.to_numeric(df["p_day"], errors="coerce")

    for c in ["reward_force_buy", "reward_free", "reward_force_sell"]:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0)

    df = df.sort_values("date").reset_index(drop=True)

    p = df["p_day"].to_numpy(float)
    r_buy = df["reward_force_buy"].to_numpy(float)
    r_free = df["reward_free"].to_numpy(float)
    r_sell = df["reward_force_sell"].to_numpy(float)

    buy_grid = np.round(np.arange(0.05, 0.3501, 0.005), 6)
    sell_grid = np.round(np.arange(0.15, 0.6001, 0.005), 6)

    pairs = np.array(
        [(b, s) for b, s in product(buy_grid, sell_grid) if s - b >= MIN_GAP],
        dtype=float,
    )

    buy_levels = pairs[:, 0]
    sell_levels = pairs[:, 1]

    rows = []

    for i in range(len(df)):
        start = max(0, i - LOOKBACK_DAYS)

        hist_p = p[start:i]
        hist_buy = r_buy[start:i]
        hist_free = r_free[start:i]
        hist_sell = r_sell[start:i]

        valid = np.isfinite(hist_p)
        hist_p = hist_p[valid]
        hist_buy = hist_buy[valid]
        hist_free = hist_free[valid]
        hist_sell = hist_sell[valid]

        if len(hist_p) < MIN_OBS:
            buy_level = 0.20
            sell_level = 0.30
            best_score = np.nan
            window_size = len(hist_p)
        else:
            buy_mask = hist_p[:, None] <= buy_levels[None, :]
            sell_mask = hist_p[:, None] >= sell_levels[None, :]

            rewards = np.where(
                buy_mask,
                hist_buy[:, None],
                np.where(sell_mask, hist_sell[:, None], hist_free[:, None]),
            )

            scores = rewards.sum(axis=0)
            best_idx = int(np.nanargmax(scores))

            buy_level = float(buy_levels[best_idx])
            sell_level = float(sell_levels[best_idx])
            best_score = float(scores[best_idx])
            window_size = len(hist_p)

        current_p = float(p[i]) if np.isfinite(p[i]) else np.nan
        daily_action = action_from_p(current_p, buy_level, sell_level)

        if daily_action == "FORCE_BUY":
            daily_reward = float(r_buy[i])
        elif daily_action == "FORCE_SELL":
            daily_reward = float(r_sell[i])
        else:
            daily_reward = float(r_free[i])

        rows.append({
            "date": df.loc[i, "date"],
            "p_day": current_p,
            "daily_buy_level": buy_level,
            "daily_sell_level": sell_level,
            "threshold_gap": sell_level - buy_level,
            "daily_action": daily_action,
            "daily_reward_from_reward_log": daily_reward,
            "score_on_lookback": best_score,
            "window_size": window_size,
            "reward_force_buy": float(r_buy[i]),
            "reward_free": float(r_free[i]),
            "reward_force_sell": float(r_sell[i]),
            "best_action_ex_post": df.loc[i, "best_action_ex_post"] if "best_action_ex_post" in df.columns else None,
        })

    return pd.DataFrame(rows)

def load_daily_closes():
    bars = pd.read_csv(FIVE_MIN_PATH)

    ts_col = next(
        (c for c in bars.columns if c.lower() in {"timestamp", "date", "datetime", "time"}),
        bars.columns[0],
    )
    close_col = next(
        (c for c in bars.columns if c.lower() in {"close", "adj close", "adj_close", "c"}),
        None,
    )

    if close_col is None:
        raise ValueError(f"{FIVE_MIN_PATH} must contain a close column")

    bars[ts_col] = pd.to_datetime(bars[ts_col], errors="coerce")
    bars[close_col] = pd.to_numeric(bars[close_col], errors="coerce")
    bars = bars.dropna(subset=[ts_col, close_col]).sort_values(ts_col)
    bars["date"] = bars[ts_col].dt.normalize()

    return bars.groupby("date")[close_col].last()

def gate_allows_trade(gate, side):
    gate = str(gate)
    side = str(side).lower()

    if gate == "FORCE_BUY" and side == "sell":
        return False
    if gate == "FORCE_SELL" and side == "buy":
        return False
    return True

def replay_old_trade_points_with_new_gate(gates):
    trades = pd.read_csv(OLD_TRADES_PATH)

    trades["exec_ts"] = pd.to_datetime(
        trades["exec_ts"] if "exec_ts" in trades.columns else trades["ts"],
        errors="coerce",
    )
    trades["ts"] = pd.to_datetime(trades["ts"], errors="coerce")
    trades["exec_px"] = pd.to_numeric(trades["exec_px"], errors="coerce")

    trades = (
        trades.dropna(subset=["exec_ts", "exec_px"])
        .sort_values("exec_ts")
        .reset_index(drop=True)
    )
    trades["date"] = trades["exec_ts"].dt.normalize()

    gate_by_day = gates.set_index("date").to_dict("index")
    closes = load_daily_closes()

    all_days = sorted(
        set(gates["date"])
        .union(set(trades["date"]))
        .union(set(closes.index))
    )

    cash = float(INITIAL_CAPITAL)
    pos = 0
    qty = 0.0
    entry_px = np.nan
    entry_ts = pd.NaT

    replay_trades = []
    daily_rows = []

    for day in all_days:
        day_trades = trades[trades["date"].eq(day)]

        for _, tr in day_trades.iterrows():
            side = str(tr["side"]).lower()
            px = float(tr["exec_px"])

            gate_info = gate_by_day.get(day, {})
            gate = gate_info.get("daily_action", "FREE")

            if not gate_allows_trade(gate, side):
                continue

            if side == "buy" and pos == 0:
                notional = cash
                spend = notional * (1.0 + FEE_PCT)

                if spend > cash:
                    spend = cash
                    notional = spend / (1.0 + FEE_PCT)

                qty = notional / px if px > 0 else 0.0
                cash -= spend
                pos = 1
                entry_px = px
                entry_ts = tr["exec_ts"]

                replay_trades.append({
                    "side": "buy",
                    "signal_ts": tr["ts"],
                    "exec_ts": tr["exec_ts"],
                    "exec_px": px,
                    "qty": qty,
                    "fee": spend - notional,
                    "gate": gate,
                    "p_day": gate_info.get("p_day", np.nan),
                    "daily_buy_level": gate_info.get("daily_buy_level", np.nan),
                    "daily_sell_level": gate_info.get("daily_sell_level", np.nan),
                    "old_pred": tr.get("pred", np.nan),
                    "old_th": tr.get("th", np.nan),
                    "old_gate": tr.get("gate", None),
                    "reason": "new daily gate allows old 5m trade point",
                })

            elif side == "sell" and pos == 1:
                proceeds = qty * px * (1.0 - FEE_PCT)
                fee = qty * px - proceeds
                pnl = proceeds - qty * float(entry_px)

                cash += proceeds

                replay_trades.append({
                    "side": "sell",
                    "signal_ts": tr["ts"],
                    "exec_ts": tr["exec_ts"],
                    "exec_px": px,
                    "qty": qty,
                    "fee": fee,
                    "gate": gate,
                    "p_day": gate_info.get("p_day", np.nan),
                    "daily_buy_level": gate_info.get("daily_buy_level", np.nan),
                    "daily_sell_level": gate_info.get("daily_sell_level", np.nan),
                    "old_pred": tr.get("pred", np.nan),
                    "old_th": tr.get("th", np.nan),
                    "old_gate": tr.get("gate", None),
                    "entry_px": entry_px,
                    "entry_ts": entry_ts,
                    "pnl": pnl,
                    "reason": "new daily gate allows old 5m trade point",
                })

                pos = 0
                qty = 0.0
                entry_px = np.nan
                entry_ts = pd.NaT

        close_px = float(closes.get(day, np.nan))
        equity = cash if pos == 0 or not np.isfinite(close_px) else cash + qty * close_px

        gate_info = gate_by_day.get(day, {})

        daily_rows.append({
            "date": day,
            "equity": equity,
            "cash": cash,
            "pos": pos,
            "qty": qty,
            "entry_px": entry_px,
            "close_px": close_px,
            "p_day": gate_info.get("p_day", np.nan),
            "daily_action": gate_info.get("daily_action", None),
            "daily_buy_level": gate_info.get("daily_buy_level", np.nan),
            "daily_sell_level": gate_info.get("daily_sell_level", np.nan),
            "threshold_gap": gate_info.get("threshold_gap", np.nan),
        })

    return pd.DataFrame(daily_rows), pd.DataFrame(replay_trades)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

gates_df = build_daily_gate_replay()
new_daily_log_df, new_trades_df = replay_old_trade_points_with_new_gate(gates_df)

gates_df.to_csv(OUTPUT_DIR / "daily_gate_replay_lookback30.csv", index=False)
new_daily_log_df.to_csv(OUTPUT_DIR / "daily_log.csv", index=False)
new_trades_df.to_csv(OUTPUT_DIR / "trades.csv", index=False)

summary_df = pd.DataFrame([{
    "initial_capital": INITIAL_CAPITAL,
    "final_equity": new_daily_log_df["equity"].dropna().iloc[-1],
    "num_trades": len(new_trades_df),
    "num_buys": int((new_trades_df["side"] == "buy").sum()) if not new_trades_df.empty else 0,
    "num_sells": int((new_trades_df["side"] == "sell").sum()) if not new_trades_df.empty else 0,
    "lookback_days": LOOKBACK_DAYS,
    "min_obs": MIN_OBS,
    "min_gap": MIN_GAP,
}])

summary_df.to_csv(OUTPUT_DIR / "summary.csv", index=False)

display(summary_df)
display(new_daily_log_df.tail(30))
display(new_trades_df.tail(30))

print("saved folder:", OUTPUT_DIR)

,initial_capital,final_equity,num_trades,num_buys,num_sells,lookback_days,min_obs,min_gap
0,100000.0,2.029732e+06,3092,1546,1546,252,60,0.02


,date,equity,cash,pos,qty,entry_px,close_px,p_day,daily_action,daily_buy_level,daily_sell_level,threshold_gap
4077,2026-04-29,1.929975e+06,0.000000e+00,1,30503.790514,62.3500,63.2700,0.181683,FORCE_BUY,0.235,0.5,0.265
4078,2026-04-30,1.938900e+06,0.000000e+00,1,30503.790514,62.3500,63.5626,0.183681,FORCE_BUY,0.235,0.5,0.265
4079,2026-05-01,1.987627e+06,0.000000e+00,1,30503.790514,62.3500,65.1600,0.218608,FORCE_BUY,0.255,0.5,0.245
4080,2026-05-04,1.988788e+06,1.988788e+06,0,0.000000,NaN,64.6608,0.213682,FREE,0.195,0.5,0.305
4081,2026-05-05,1.988788e+06,1.988788e+06,0,0.000000,NaN,69.1000,0.234111,FREE,0.195,0.5,0.305
4082,2026-05-06,1.979624e+06,0.000000e+00,1,27795.785258,71.5500,71.2203,0.187017,FORCE_BUY,0.195,0.5,0.305
4083,2026-05-07,2.030991e+06,0.000000e+00,1,28493.142309,70.5500,71.2800,0.242005,FREE,0.195,0.5,0.305
4084,2026-05-08,2.102827e+06,2.102827e+06,0,0.000000,NaN,76.5500,0.257331,FREE,0.195,0.5,0.305
4085,2026-05-11,2.118337e+06,0.000000e+00,1,27525.167231,76.9500,76.9600,0.252784,FREE,0.195,0.5,0.305
4086,2026-05-12,2.049740e+06,0.000000e+00,1,27616.710290,75.1700,74.2210,0.216910,FREE,0.195,0.5,0.305


,side,signal_ts,exec_ts,exec_px,qty,fee,gate,p_day,daily_buy_level,daily_sell_level,old_pred,old_th,old_gate,reason,entry_px,entry_ts,pnl
3062,buy,2026-05-18 14:10:00,2026-05-18 14:15:00,72.8788,30188.242851,0.0,FREE,0.229691,0.195,0.5,12.667041,-0.5,FREE,new daily gate allows old 5m trade point,NaN,NaT,NaN
3063,sell,2026-05-19 09:30:00,2026-05-19 09:35:00,73.2200,30188.242851,0.0,FREE,0.237266,0.195,0.5,7.572336,-0.5,FREE,new daily gate allows old 5m trade point,72.8788,2026-05-18 14:15:00,10300.228461
3064,buy,2026-05-19 13:50:00,2026-05-19 13:55:00,73.8300,29938.820826,0.0,FREE,0.237266,0.195,0.5,10.105450,-0.5,FREE,new daily gate allows old 5m trade point,NaN,NaT,NaN
3065,sell,2026-05-19 18:05:00,2026-05-19 18:10:00,73.1800,29938.820826,0.0,FREE,0.237266,0.195,0.5,7.119667,-0.5,FREE,new daily gate allows old 5m trade point,73.8300,2026-05-19 13:55:00,-19460.233537
3066,buy,2026-05-20 05:05:00,2026-05-20 05:10:00,74.1900,29531.242864,0.0,FREE,0.209540,0.195,0.5,8.697261,-0.5,FREE,new daily gate allows old 5m trade point,NaN,NaT,NaN
3067,sell,2026-05-20 05:45:00,2026-05-20 05:50:00,74.3700,29531.242864,0.0,FREE,0.209540,0.195,0.5,5.563221,-0.5,FREE,new daily gate allows old 5m trade point,74.1900,2026-05-20 05:10:00,5315.623715
3068,buy,2026-05-21 06:25:00,2026-05-21 06:30:00,75.6300,29039.250717,0.0,FORCE_BUY,0.190151,0.195,0.5,11.668165,-0.5,FORCE_BUY,new daily gate allows old 5m trade point,NaN,NaT,NaN
3069,sell,2026-05-22 04:05:00,2026-05-22 04:10:00,78.1500,29039.250717,0.0,FREE,0.201362,0.195,0.5,7.864723,-0.5,FREE,new daily gate allows old 5m trade point,75.6300,2026-05-21 06:30:00,73178.911808
3070,buy,2026-05-22 05:00:00,2026-05-22 05:05:00,77.7200,29199.915640,0.0,FREE,0.201362,0.195,0.5,4.870263,-0.5,FREE,new daily gate allows old 5m trade point,NaN,NaT,NaN
3071,sell,2026-05-22 07:00:00,2026-05-22 07:05:00,77.3200,29199.915640,0.0,FREE,0.201362,0.195,0.5,6.535097,-0.5,FREE,new daily gate allows old 5m trade point,77.7200,2026-05-22 05:05:00,-11679.966256


saved folder: output_replay_new_daily_gate_old_5m_signals_lookback30


In [3]:
from adaptive_reward_checkpoint_fresh import run_adaptive_reward_from_snapshot

checkpoint_path = r"output_adaptive_reward_snapshot_build_TQQQ_252days_at_2020\year_start_checkpoints\final_checkpoint__start_2020.joblib"

res = run_adaptive_reward_from_snapshot(
    snapshot_path=checkpoint_path,

    # Simulate through this local-data timestamp.
    end_time="2026-06-12 16:00:00",

    # Actual trading begins here.
    # The first trading day will use the latest prior p_day/gate from the checkpoint/history.
    trade_start="2020-01-01 09:30:00",

    # Optional; defaults to checkpoint_time + 1 day.
    # Keep this at or before trade_start if you want all trade_start signals considered.
    sim_start="2020-01-01 09:30:00",

    output_dir=r"output_prev_checkpoint_local_TQQQ_20200101_20260612",
    save_snapshot_path=r"output_prev_checkpoint_local_TQQQ_20200101_20260612\continued_checkpoint.joblib",

    # Start clean/flat from this capital.
    reset_execution_state=True,
    initial_capital=100000.0,
    fee_pct=0.0,

    # Keep saved model/checkpoint logic.
    autosave_year_start_checkpoints=False,
    verbose=True,
)

print("Output dir:", res["output_dir"])
print("Continued checkpoint:", res["continued_snapshot_path"])
print(res["daily_reward_df"].tail())
print(res["signal_decisions_all_df"].tail())

[TRAIN][DAILY-PROB] n=5523 pos=1179 (21.35%)
[TRAIN][5M] asof=2020-01-02 feats=69 buy=YES sell=YES rows=57820
[TRAIN][5M] asof=2020-01-07 feats=69 buy=YES sell=YES rows=57927
[TRAIN][5M] asof=2020-01-13 feats=69 buy=YES sell=YES rows=58140
[TRAIN][5M] asof=2020-01-21 feats=69 buy=YES sell=YES rows=58318
[TRAIN][5M] asof=2020-01-27 feats=69 buy=YES sell=YES rows=58465
[TRAIN][5M] asof=2020-02-03 feats=69 buy=YES sell=YES rows=58676
[TRAIN][5M] asof=2020-02-10 feats=69 buy=YES sell=YES rows=58878
[TRAIN][5M] asof=2020-02-18 feats=69 buy=YES sell=YES rows=59038
[TRAIN][5M] asof=2020-02-24 feats=69 buy=YES sell=YES rows=59189
[TRAIN][5M] asof=2020-03-02 feats=69 buy=YES sell=YES rows=59426
[TRAIN][5M] asof=2020-03-09 feats=69 buy=YES sell=YES rows=59631
[TRAIN][5M] asof=2020-03-16 feats=69 buy=YES sell=YES rows=59895
[TRAIN][5M] asof=2020-03-23 feats=69 buy=YES sell=YES rows=60104
[TRAIN][5M] asof=2020-03-30 feats=69 buy=YES sell=YES rows=60283
[TRAIN][5M] asof=2020-04-06 feats=69 buy=YES 

In [3]:
from adaptive_reward_checkpoint_fresh import run_daily_p_values_fresh

res = run_daily_p_values_fresh(
    daily_csv_path=r"DataAPI/data/TQQQ_day.csv",
    k5m_csv_path=r"DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",

    start_time="2012-01-01",
    end_time="2026-06-12 16:00:00",

    # Set equal to start_time for truly fresh daily-only run.
    # Or set earlier if you want warmup/training history before the comparison window.
    daily_chan_start="2012-01-01",
    accumulation_start="2012-01-01",

    output_dir=r"output_TQQQ_daily_p_only_fresh_2012_20260612",
    verbose=True,
)

p_df = res["p_df"]
print(res["output_csv_path"])
display(p_df.tail())

[DAILY-P-FRESH] code=TQQQ daily_chan_start=2012-01-01 start_time=2012-01-01 end_time=2026-06-12 16:00:00
[TRAIN][DAILY-PROB] n=200 pos=50 (25.00%)
[TRAIN][DAILY-PROB] n=225 pos=53 (23.56%)
[TRAIN][DAILY-PROB] n=250 pos=57 (22.80%)
[TRAIN][DAILY-PROB] n=275 pos=62 (22.55%)
[TRAIN][DAILY-PROB] n=300 pos=63 (21.00%)
[TRAIN][DAILY-PROB] n=325 pos=66 (20.31%)
[TRAIN][DAILY-PROB] n=350 pos=71 (20.29%)
[TRAIN][DAILY-PROB] n=375 pos=72 (19.20%)
[TRAIN][DAILY-PROB] n=400 pos=82 (20.50%)
[TRAIN][DAILY-PROB] n=425 pos=90 (21.18%)
[TRAIN][DAILY-PROB] n=450 pos=90 (20.00%)
[TRAIN][DAILY-PROB] n=475 pos=91 (19.16%)
[TRAIN][DAILY-PROB] n=500 pos=98 (19.60%)
[TRAIN][DAILY-PROB] n=525 pos=105 (20.00%)
[TRAIN][DAILY-PROB] n=550 pos=110 (20.00%)
[TRAIN][DAILY-PROB] n=575 pos=116 (20.17%)
[TRAIN][DAILY-PROB] n=600 pos=122 (20.33%)
[TRAIN][DAILY-PROB] n=625 pos=124 (19.84%)
[TRAIN][DAILY-PROB] n=650 pos=131 (20.15%)
[TRAIN][DAILY-PROB] n=675 pos=137 (20.30%)
[TRAIN][DAILY-PROB] n=700 pos=141 (20.14%)
[TRAI

,timestamp,date,p_day,dp_vs_minK,dp_vs_maxK,source
3627,2026-06-08,2026-06-08,0.272320,0.049486,-0.123088,fresh_daily
3628,2026-06-09,2026-06-09,0.383073,0.160239,-0.012334,fresh_daily
3629,2026-06-10,2026-06-10,0.305193,0.051424,-0.090214,fresh_daily
3630,2026-06-11,2026-06-11,0.279197,0.025428,-0.116211,fresh_daily
3631,2026-06-12,2026-06-12,0.251750,-0.020570,-0.143658,fresh_daily


In [ ]:
from next_day_gate_model import (
    next_day_gate_config_from_adaptive_kwargs,
    run_next_day_gate_experiment,
)
from adaptive_trade_extensions import RollingThresholdConfig, make_threshold_grid

common_kwargs = dict(
    daily_csv_path="DataAPI/data/TQQQ_day.csv",
    k5m_csv_path="DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    daily_chan_start="2012-01-01",
    accumulation_start="2018-01-01",
    N_confirm=5,  # ignored by new direct gate model
    min_labeled_days_to_train=200,  # ignored by new direct gate model
    retrain_every_new_labels=25,  # ignored by new direct gate model
    dp_lookback=5,  # ignored by new direct gate model
    lookahead_days_5m=2.0,
    retrain_every_days_5m=5,
    min_samples_total_5m=300,
    threshold_window_days=2.0,
    threshold_ret_grid=None,
    threshold_min_open_signals=10,
    initial_capital=100000.0,
    fee_pct=0.0,
    daily_chan_max_klines=500,
    five_chan_max_klines=500,
    macro_files={
        "vix_": "VIX.csv",
        "US2Y": "US2Y.csv",
        "US10Y": "US10Y.csv",
    },
    static_buy_level=0.20,  # ignored by new direct gate model
    static_sell_level=0.30,  # ignored by new direct gate model
    daily_threshold_config=RollingThresholdConfig(  # ignored by new direct gate model
        lookback_days=252,
        buy_grid=make_threshold_grid(0.05, 0.35, 0.005),
        sell_grid=make_threshold_grid(0.15, 0.60, 0.005),
        min_gap=0.02,
        min_obs=60,
        switch_penalty=0.0,
    ),
    verbose=True,
)

cfg = next_day_gate_config_from_adaptive_kwargs(
    common_kwargs,
    trade_signals_csv="trade_signals_tqqq.csv",
    start_year=2020,
    end_year=2026,
    output_dir="output_next_day_gate_model_TQQQ_signals",
    checkpoint_dir="checkpoints/next_day_gate_model_TQQQ_signals",
    model_type="extra_trees",
    min_train_days=252,
    retrain_every_n_days=20,
    checkpoint_every_n_days=20,
)

result = run_next_day_gate_experiment(cfg)

print("daily predictions:", result["predictions_path"])
print("daily log:", result["daily_log_path"])
print("executed trades:", result["executed_trades_path"])
print("yearly summary:", result["summary_path"])
print("checkpoint:", result["checkpoint_path"])

display(result["summary"].tail())
display(result["daily_log"].tail())
display(result["executed_trades"].tail())

TypeError: NextDayGateConfig.__init__() got an unexpected keyword argument 'trade_signals_csv'

In [3]:
from pathlib import Path
import pandas as pd
from plot_bspoints import build_bspoints_from_kline, plot_bspoints
price_df, bsp_df = build_bspoints_from_kline(
    "DataAPI/data/TQQQ_5M.csv",
    code="TQQQ",
    freq="5m",
    start=pd.Timestamp("2024-05-01"),
    end=pd.Timestamp("2024-07-31"),
    max_klines=500,
    warmup_bars=500,
)
plot_output = Path("outputs/bspoints_TQQQ_5m_2024.05-07.png")
excel_output = Path("outputs/bspoints_TQQQ_5m_2024.05-07.xlsx")
plot_output.parent.mkdir(parents=True, exist_ok=True)
plot_bspoints(
    price_df,
    bsp_df,
    title="TQQQ 5min BSP points",
    output=plot_output,
    annotate=False,  # Shows BSP type, buy/sell, Bi direction and segment direction
    show_volume=False,
)
bsp_df.to_excel(
    excel_output,
    sheet_name="BSP Points",
    index=False,
)
print(
    bsp_df[
        [
            "timestamp",
            "bsp_type",
            "direction",
            "bi_direction",
            "segment_direction",
        ]
    ].head()
)
print(f"Plot saved: {plot_output.resolve()}")
print(f"Excel saved: {excel_output.resolve()}")

            timestamp bsp_type direction bi_direction segment_direction
0 2024-05-01 04:00:00       1p       buy         down              down
1 2024-05-01 05:05:00       1p       buy         down              down
2 2024-05-01 05:10:00       1p       buy         down              down
3 2024-05-01 05:30:00       1p       buy         down              down
4 2024-05-01 06:25:00       1p       buy         down              down
Plot saved: C:\Users\TonyTang\Documents\chan.py\outputs\bspoints_TQQQ_5m_2024.05-07.png
Excel saved: C:\Users\TonyTang\Documents\chan.py\outputs\bspoints_TQQQ_5m_2024.05-07.xlsx
